In [ ]:
import numpy as np
import pandas as pd
import pyabf
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tsfel
import warnings
import os
import umap  

In [ ]:
output_dir = Path("analyte_signature_complete_analysis")
output_dir.mkdir(exist_ok=True)

file_pairs = [
    {
        'classification_csv': '/home/rushang_phira/src/data/classified/20200309_wtAeL_4M_KCl_1L_1X_L2AS4_AS5_AS6_AS7_AS8_AS9_AS10.abf_currentlevels_classified.csv',
        'abf_file': '/mnt/vnas/ml4nanopore_data/raw/PeptideLadders/Ladder2/20200309 wtAeL 4M KCl 1µL 1X L2AS4_AS5_AS6_AS7_AS8_AS9_AS10.abf'
    },
    # {
    #     'classification_csv': '/home/rushang_phira/src/data/classified/2020-03-13_wtAeL_L1AS4_AS5_AS6_AS7_AS8_AS9_AS10.abf_currentlevels_classified.csv',
    #     'abf_file': '/mnt/vnas/ml4nanopore_data/raw/PeptideLadders/Ladder1/2020-03-13 wtAeL L1AS4 AS5 AS6 AS7 AS8 AS9 AS10.abf'
    # },
    # {
    #     'classification_csv': '/home/rushang_phira/src/data/classified/feature_set/tsfresh/mdr30mv.csv',
    #     'abf_file': '/mnt/vnas/ml4nanopore_data/raw/priyanka_data/MDRprotein/14.09.2023 4M Kcl wt-AeL MDR 30mV.abf'
    # },
    # {
    #     'classification_csv': '/home/rushang_phira/src/data/classified/feature_set/tsfresh/app30mv.csv',
    #     'abf_file': '/mnt/vnas/ml4nanopore_data/raw/priyanka_data/aplysia_peptides/l-AP and d-AP 4M Kcl R220S -30mV.abf'
    # },
    # {
    #     'classification_csv': '/home/rushang_phira/src/data/classified/2021-03-19_3.5_L_H4Ac_K8_K12_K16-2.abf_currentlevels.15_classified.csv',
    #     'abf_file': '/mnt/vnas/ml4nanopore_data/processed/2021-03-19 AeL R220S Acetylated Peptides ABF/2021-03-19 3.5 µL H4Ac K8 K12 K16-2.abf'
    # }
    
]

# Sampling parameters
RANDOM_STATE = 42
MAX_EVENT_LENGTH = 1000
BASELINE_SEGMENT_LENGTH = 1000
MAX_BASELINE_SEGMENTS_PER_FILE = 2000

In [ ]:
from scipy.optimize import bisect
# Balanced Sampling + Clustering + Supervised Analysis
def soft_threshold(x, delta):
    """Soft-thresholding operator"""
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0)

def compute_cer(true_labels, cluster_labels):
    """Compute classification error rate"""
    cm = confusion_matrix(true_labels, cluster_labels)
    return 1 - np.trace(cm) / np.sum(cm)

def sparse_kmeans(X, K, s, true_labels=None, max_iter=1000, tol=1e-6, patience=50, min_iter=10, random_state=42):
    """
    Sparse K-means clustering following Witten & Tibshirani (2010)
    
    Parameters:
    - X: numpy array of shape (n, p)
    - K: int, number of clusters
    - s: float, L1 bound on weights (1 <= s <= sqrt(p))
    - true_labels: optional true labels for evaluation
    - max_iter: int, maximum iterations
    - tol: float, convergence tolerance
    - random_state: random seed
    
    Returns:
    - cluster_labels: array of cluster assignments
    - weights: array of feature weights
    - metrics: dictionary of performance metrics
    """
    n, p = X.shape
    
    # validate parameters
    if s < 1 or s > np.sqrt(p):
        raise ValueError(f"s must be between 1 and sqrt(p)={np.sqrt(p):.2f}")
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # weight initialization
    weights = np.ones(p) / np.sqrt(p)
    objective_history = []
    weight_history = [weights.copy()]

    # Early stopping variables
    best_weights = weights.copy()
    best_objective = -np.inf
    patience_counter = 0
    converged = False
    print(f"{n} samples, {p} features, s={s}")
    
    for iteration in range(max_iter):
        # Weighted K-means clustering
        weighted_X = X_scaled * np.sqrt(weights)
        kmeans = KMeans(n_clusters=K, random_state=random_state, n_init=10)
        cluster_labels = kmeans.fit_predict(weighted_X)
        
        # Compute BCSS
        BCSS = np.zeros(p)
        for j in range(p):
            # Total sum of squares for feature j
            overall_mean = np.mean(X_scaled[:, j])
            total_ss = np.sum((X_scaled[:, j] - overall_mean) ** 2)
            
            # Within cluster sum of squares for feature j
            within_ss = 0
            for k in range(K):
                cluster_indices = (cluster_labels == k)
                n_k = np.sum(cluster_indices)
                if n_k > 0:  # Avoid empty clusters
                    cluster_mean = np.mean(X_scaled[cluster_indices, j])
                    within_ss += np.sum((X_scaled[cluster_indices, j] - cluster_mean) ** 2)
            
            # BCSS = Total SS - Within SS
            BCSS[j] = (total_ss - within_ss)
        
        # Update weights with L1 constraint enforcement
        a = BCSS
        
        # Scale BCSS to unit norm for numerical stability
        a = a / (np.linalg.norm(a, 2) + 1e-10)
        
        # Binary search for delta to satisfy ||w||₁ = s
        def l1_norm(delta):
            w_temp = soft_threshold(a, delta)
            norm_w = np.linalg.norm(w_temp, 2)
            if norm_w > 0:
                w_temp = w_temp / norm_w
            return np.sum(np.abs(w_temp)) - s
        
        # Check if constraint is already satisfied
        current_l1 = np.sum(np.abs(a / (np.linalg.norm(a, 2) + 1e-10)))
        
        if current_l1 <= s:
            delta = 0
            w_new = a
        else:
            # Binary search for optimal delta
            delta_low, delta_high = 0, np.max(np.abs(a))
            delta = bisect(l1_norm, delta_low, delta_high, xtol=1e-8, maxiter=50)
            w_new = soft_threshold(a, delta)
        
        # Normalize final weights
        norm_w = np.linalg.norm(w_new, 2)
        if norm_w > 0:
            w_new = w_new / norm_w
        else:
            w_new = np.ones(p) / np.sqrt(p)
        
        # Calculate objective
        current_objective = np.sum(w_new * BCSS)
        objective_history.append(current_objective)
        weight_history.append(w_new.copy())

        # Check for improvement
        if current_objective > best_objective:
            best_objective = current_objective
            best_weights = w_new.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        # Check convergence
        weight_change = np.linalg.norm(w_new - weights, 2)
        n_nonzero = np.sum(w_new > 1e-6)
        
        print(f"Iteration {iteration + 1}:")
        print(f"  - BCSS range: [{BCSS.min():.6f}, {BCSS.max():.6f}]")
        print(f"  - Weight change: {weight_change:.6f}")
        print(f"  - Objective: {current_objective:.6f}")
        print(f"  - Non-zero features: {n_nonzero}/{p} ({n_nonzero/p*100:.1f}%)")
        
        # Early stopping conditions
        if weight_change < tol and iteration >= min_iter:
            print(f"converged after {iteration + 1} iterations (weight change < {tol})")
            converged = True
            break
        elif patience_counter >= patience and iteration >= min_iter:
            print(f"early stopping after {iteration + 1} iterations (no improvement for {patience} iterations)")
            converged = True
            break
            
        weights = w_new
    if not converged:
            print(f"reached maximum iterations ({max_iter})")
    
    # Use best weights found
    final_weights = best_weights
    final_n_nonzero = np.sum(final_weights > 1e-6)
    
    print(f"Final: {final_n_nonzero} non-zero features, Best objective: {best_objective:.6f}")
    
    return cluster_labels, final_weights, objective_history


def extract_baseline_segments(abf_data, event_regions, num_segments=100):
    """Extract baseline segments avoiding event regions"""
    baseline_segments = []
    total_length = len(abf_data.data[0])
    
    for attempt in range(num_segments * 10):
        start_idx = np.random.randint(0, total_length - BASELINE_SEGMENT_LENGTH)
        end_idx = start_idx + BASELINE_SEGMENT_LENGTH
        
        # Check if this segment overlaps with any event
        overlap = False
        for event_start, event_end in event_regions:
            if not (end_idx < event_start or start_idx > event_end):
                overlap = True
                break
        
        if not overlap:
            baseline_signal = abf_data.data[0][start_idx:end_idx]
            baseline_segments.append(baseline_signal)
            
        if len(baseline_segments) >= num_segments:
            break
            
    return baseline_segments

def extract_tsfresh_features(signal, series_id):
    """Extract features using tsfresh"""
    from tsfresh import extract_features
    
    df = pd.DataFrame({
        "series_id": series_id,
        "time": np.arange(len(signal)),
        "value": signal
    })
    
    features = extract_features(
        df, 
        column_id="series_id",
        column_sort="time", 
        column_value="value",
        disable_progressbar=True
    )
    return features

def extract_tsfel_features(signal, segment_id):
    """Extract features with tsfel"""
    try:
        # Ensure signal is long enough
        if len(signal) < 20:
            print(f"Signal too short for {segment_id}: {len(signal)} samples")
            return None
 
        df = pd.DataFrame({"signal": signal})
        cfg = tsfel.get_features_by_domain()
        
        # extract features
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            features = tsfel.time_series_features_extractor(cfg, df, verbose=0)
        
        # check for features extracted successfully
        if features is None or features.empty:
            print(f"    No features extracted for {segment_id}")
            return None
        
        # add segment id as index
        features = features.reset_index(drop=True)
        features['segment_id'] = segment_id
        
        return features
        
    except Exception as e:
        # print(f"    Error extracting TSFEL features for {segment_id}: {e}")
        return None

def create_balanced_dataset_for_abf_tsfresh(abf_path, classification_csv_path):
    """
    creating here a balanced dataset with baseline and event segments for one ABF file
    """
    
    classified_data = pd.read_csv(classification_csv_path, sep='\t')
    
    if len(classified_data) == 0:
        print(f"No matching events found for {abf_path}")
        return None
        
    try:
        abf_data = pyabf.ABF(abf_path)
    except Exception as e:
        print(f"error with loading abf {abf_path}: {e}")
        return None
    
    # get event regions because we want to avoid baseline
    event_regions = []
    for _, row in classified_data.iterrows():
        event_regions.append((int(row["idxstart"]), int(row["idxend"])))
    
    # first we sample baseline segments (balanced with events)
    num_baseline_segments = min(len(classified_data), MAX_BASELINE_SEGMENTS_PER_FILE)
    baseline_segments = extract_baseline_segments(abf_data, event_regions, num_baseline_segments)
    
    all_segments = []
    segment_info = []
    
    # here is the baseline extraction process
    for i, baseline_signal in enumerate(baseline_segments):
        features = extract_tsfel_features(baseline_signal, f"baseline_{i}")
        features['true_label'] = 'baseline'
        features['analyte_label'] = 'baseline'
        features['abf_file'] = os.path.basename(abf_path)
        features['segment_type'] = 'baseline'
        all_segments.append(features)
        segment_info.append({'type': 'baseline', 'file': abf_path})
    
    # we get the events here
    for i, row in classified_data.iterrows():
        s, e = int(row["idxstart"]), int(row["idxend"])
        event_signal = abf_data.data[0][s:e]
        
        # limit segment length for efficiency
        if len(event_signal) > MAX_EVENT_LENGTH:
            event_signal = event_signal[:MAX_EVENT_LENGTH]
        
        features = extract_tsfel_features(event_signal, f"event_{i}")
        features['true_label'] = row['final_label']
        features['analyte_label'] = row['final_label']
        features['abf_file'] = os.path.basename(abf_path)
        features['segment_type'] = 'event'
        all_segments.append(features)
        segment_info.append({'type': 'event', 'file': abf_path, 'analyte': row['final_label']})
    
    if all_segments:
        combined_features = pd.concat(all_segments, ignore_index=True)
        print(f"  Extracted {len(baseline_segments)} baseline + {len(classified_data)} event segments")
        return combined_features
    else:
        return None
    
def create_balanced_dataset_for_abf(abf_path, classification_csv_path):
    """
    Create balanced dataset with baseline and event segments for one ABF file
    """
    
    classified_data = pd.read_csv(classification_csv_path)
    
    print(f"Loaded {len(classified_data)} events from csv")
    
    if len(classified_data) == 0:
        print(f"No matching events found for {abf_path}")
        return None
        
    try:
        abf_data = pyabf.ABF(abf_path)
        print(f"  Successfully loaded ABF file")
    except Exception as e:
        print(f"Error loading ABF file {abf_path}: {e}")
        return None
    
    # get event regions for baseline avoidance
    event_regions = []
    for _, row in classified_data.iterrows():
        event_regions.append((int(row["idxstart"]), int(row["idxend"])))
    
    # sample baseline segments (balance with events)
    num_baseline_segments = min(len(classified_data), MAX_BASELINE_SEGMENTS_PER_FILE)
    baseline_segments = extract_baseline_segments(abf_data, event_regions, num_baseline_segments)
    
    print(f"  Extracted {len(baseline_segments)} baseline segments")
    
    all_features = []
    
    # extract baseline features with tsfel
    baseline_count = 0
    for i, baseline_signal in enumerate(baseline_segments):
        features = extract_tsfel_features(baseline_signal, f"baseline_{i}")
        if features is not None and not features.empty:
            # Add metadata
            # features['true_label'] = f"b_{abf_path[45:60]}"
            # features['analyte_label'] = f"b_{abf_path[45:60]}"
            features['true_label'] = f"baseline"
            features['analyte_label'] = f"baseline"
            features['abf_file'] = os.path.basename(abf_path)
            features['segment_type'] = 'baseline'
            features['segment_length'] = len(baseline_signal)
            all_features.append(features)
            baseline_count += 1
    
    # extract event features using tsfel
    event_count = 0
    
    for i, row in classified_data.iterrows():
        s, e = int(row["idxstart"]), int(row["idxend"])
        event_signal = abf_data.data[0][s:e]
        
        features = extract_tsfel_features(event_signal, f"event_{i}")
        if features is not None and not features.empty:
            # add metadata
            features['true_label'] = row['final_label']
            features['analyte_label'] = row['final_label']
            features['abf_file'] = os.path.basename(abf_path)
            features['segment_type'] = 'event'
            features['segment_length'] = len(event_signal)
            features['log_length'] = np.log(len(event_signal))
            all_features.append(features)
            event_count += 1
    
    print(f"processed: {baseline_count} baseline + {event_count} event segments")
    
    if all_features:
        combined_features = pd.concat(all_features, ignore_index=True)
        print(f"Total features shape: {combined_features.shape}")
        return combined_features
    else:
        print(f"No features extracted successfully")
        return None
    
def stratified_sample_by_analyte(feature_df, max_per_analyte=2000):
    """
    stratified sampling to balance analytes and prevent over-representation
    """
    sampled_dfs = []
    
    for analyte in feature_df['analyte_label'].unique():
        analyte_data = feature_df[feature_df['analyte_label'] == analyte]
        
        # Sample to balance classes
        n_sample = min(len(analyte_data), max_per_analyte)
        sampled = analyte_data.sample(n=n_sample, random_state=RANDOM_STATE)
        sampled_dfs.append(sampled)
    
    return pd.concat(sampled_dfs, ignore_index=True)

import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
import numpy as np

def perform_clustering_analysis(feature_df):
    """
    clustering to discover natural structure
    """
    print("\n" + "="*50)
    print("Exploratory Clustering")
    print("="*50)
    
    # Prepare data for clustering
    X = feature_df.select_dtypes(include=[np.number])
    X = X.loc[:, X.notna().all(axis=0)]
    X = X.loc[:, X.var(axis=0) > 1e-12]
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Get true labels
    true_labels = feature_df['analyte_label']
    le = LabelEncoder()
    y_true = le.fit_transform(true_labels)
    n_true_clusters = len(le.classes_)
    
    print(f"True classes: {le.classes_}")
    print(f"Performing K-means with {n_true_clusters} clusters...")
    
    # Perform clustering, after determining optimal s. Found here by trial
    # kmeans = KMeans(n_clusters=n_true_clusters, random_state=RANDOM_STATE, n_init=10)
    # cluster_labels = kmeans.fit_predict(X_scaled)
    cluster_labels, weights = sparse_kmeans(
    X=X_scaled,
    K=n_true_clusters,
    s=2.0,
    max_iter=100,
    random_state=RANDOM_STATE
    )
    
    cluster_to_true_label = {}
    for cluster in np.unique(cluster_labels):
        # Get indices of rows in this cluster
        cluster_indices = np.where(cluster_labels == cluster)[0]
        
        # Get the true labels of these points
        true_labels_in_cluster = y_true[cluster_indices]
        
        # Find the most frequent true label for this cluster
        most_common_true_label = np.bincount(true_labels_in_cluster).argmax()
        
        # Map this cluster to the true label
        cluster_to_true_label[cluster] = most_common_true_label

    # clusters matching to true labels ---
    correct_predictions = 0
    total_predictions = len(y_true)
    for cluster, true_label in cluster_to_true_label.items():
        # Find indices of points in this cluster
        cluster_indices = np.where(cluster_labels == cluster)[0]
        
        # Get the predicted labels for this cluster
        predicted_labels_in_cluster = cluster_labels[cluster_indices]
        
        # Count how many were correctly predicted
        correct_predictions += np.sum(predicted_labels_in_cluster == true_label)

    accuracy = correct_predictions / total_predictions
    print(f"Accuracy of clustering: {accuracy:.3f}")

    for cluster, true_label in cluster_to_true_label.items():

        cluster_indices = np.where(cluster_labels == cluster)[0]
        true_labels_in_cluster = y_true[cluster_indices]
        cluster_accuracy = np.sum(true_labels_in_cluster == true_label) / len(true_labels_in_cluster)
        print(f"Cluster {cluster} (mapped to {le.classes_[true_label]}): {cluster_accuracy:.3f} accuracy")

    feature_names = X.columns
    sorted_indices = np.argsort(weights)[::-1]
    sorted_indices = sorted_indices[:10]
    sorted_weights = weights[sorted_indices]
    sorted_feature_names = np.array(feature_names)[sorted_indices] if feature_names is not None else np.arange(len(weights))[sorted_indices]

    plt.figure(figsize=(10, 10))
    plt.barh(sorted_feature_names, sorted_weights)
    plt.xlabel('Feature Weight')
    plt.ylabel('Feature Index')
    plt.title('Feature Weights Visualization')
    plt.tight_layout()
    plt.show()

    import shap
    rf_model = RandomForestClassifier(random_state=RANDOM_STATE)
    rf_model.fit(X_scaled, y_true)
    
    # clustering visualization
    plt.figure(figsize=(15, 5))
    
    pca = umap.UMAP(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    plt.subplot(1, 3, 1)
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='tab10', alpha=0.6)
    
    # Manually create the legend with true labels
    handles, labels = scatter.legend_elements()
    labels = [le.classes_[int(label)] for label in range(len(le.classes_))]  # Mapping to true labels directly
    plt.legend(handles, labels, title='True Labels', loc='center left', bbox_to_anchor=(1, 0.5))
    
    plt.title('True Labels\n(UMAP Projection)')
    plt.xlabel(f'UMAP Projection 1')
    plt.ylabel(f'UMAP Projection 1')
    
    plt.subplot(1, 3, 2)
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='tab10', alpha=0.6)

    plt.title('K-means Clusters\n(UMAP Projection)')
    plt.xlabel(f'UMAP Projection 1')
    plt.ylabel(f'UMAP Projection 2')

    
    plt.tight_layout()
    plt.savefig(output_dir / 'clustering_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'cluster_labels': cluster_labels,
        'true_labels': y_true,
        'feature_matrix': X_scaled,
        'feature_names': X.columns,
        'ari': adjusted_rand_score(y_true, cluster_labels),
        'weights': weights
    }


def perform_supervised_analysis(feature_df):
    """
    Supervised analysis to identify discriminative features
    """
    print("\n")
    print("SUPERVISED FEATURE ANALYSIS")
    
    # Prepare data
    X = feature_df.select_dtypes(include=[np.number])
    X = X.loc[:, X.notna().all(axis=0)]
    X = X.loc[:, X.var(axis=0) > 1e-12]
    
    # Create multiple classification tasks
    
    # Task 1: Event vs Baseline (binary)
    y_binary = (feature_df['segment_type'] == 'event').astype(int)
    
    # Task 2: Multi-class analyte discrimination
    le = LabelEncoder()
    y_multiclass = le.fit_transform(feature_df['analyte_label'])
    class_names = le.classes_
    
    print(f"Binary classification: {np.sum(y_binary==0)} baseline vs {np.sum(y_binary==1)} events")
    print(f"Multi-class: {len(class_names)} classes - {class_names}")
    
    # Train Random Forest for binary classification (event vs baseline)
    from sklearn.model_selection import train_test_split, cross_val_score
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=0.3, random_state=RANDOM_STATE, stratify=y_binary
    )
    
    rf_binary = RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    rf_binary.fit(X_train, y_train)
    y_pred_binary = rf_binary.predict(X_test)
    binary_accuracy = accuracy_score(y_test, y_pred_binary)
    
    cv_scores = cross_val_score(rf_binary, X, y_binary, cv=5)
    
    print(f"Binary classification accuracy: {binary_accuracy:.3f}")
    print(f"Cross-validation scores: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    
    # feature importance for binary task
    binary_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_binary.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 features for Event vs Baseline discrimination:")
    print(binary_importance.head(10))
    
    # Train Random Forest for multi-class analyte discrimination
    X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
        X, y_multiclass, test_size=0.3, random_state=RANDOM_STATE, stratify=y_multiclass
    )
    
    rf_multi = RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    rf_multi.fit(X_train_multi, y_train_multi)
    y_pred_multi = rf_multi.predict(X_test_multi)
    multi_accuracy = accuracy_score(y_test_multi, y_pred_multi)
    
    print(f"\nMulti-class analyte discrimination accuracy: {multi_accuracy:.3f}")
    
    # Get feature importance for multi-class task
    multi_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_multi.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 features for analyte discrimination:")
    print(multi_importance.head(10))
    
    # Create supervised analysis visualizations
    plt.figure(figsize=(15, 5))
    
    # Plot 1: Binary feature importance
    plt.subplot(1, 3, 1)
    top_binary = binary_importance.head(10)
    plt.barh(range(len(top_binary)), top_binary['importance'].values)
    plt.yticks(range(len(top_binary)), [f[:20] for f in top_binary['feature']])
    plt.title('Top Features: Event vs Baseline')
    plt.xlabel('Importance')
    
    # Plot 2: Multi-class feature importance
    plt.subplot(1, 3, 2)
    top_multi = multi_importance.head(10)
    plt.barh(range(len(top_multi)), top_multi['importance'].values)
    plt.yticks(range(len(top_multi)), [f[:20] for f in top_multi['feature']])
    plt.title('Top Features: Analyte Discrimination')
    plt.xlabel('Importance')
    
    # Plot 3: Confusion matrix for multi-class
    plt.subplot(1, 3, 3)
    cm = confusion_matrix(y_test_multi, y_pred_multi)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Multi-class Confusion Matrix\n(Accuracy: {multi_accuracy:.3f})')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'supervised_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {
        'binary_accuracy': binary_accuracy,
        'multi_accuracy': multi_accuracy,
        'binary_importance': binary_importance,
        'multi_importance': multi_importance,
        'cv_scores': cv_scores,
        'class_names': class_names
    }

def compare_clustering_supervised_results(clustering_results, supervised_results):
    """
    Compare unsupervised clustering vs supervised results
    """
    print("METHOD COMPARISON")
    
    # Compare feature importance between clustering and supervised
    clustering_features = clustering_results['feature_names']
    supervised_binary_top = supervised_results['binary_importance'].head(20)['feature'].values
    supervised_multi_top = supervised_results['multi_importance'].head(20)['feature'].values
    
    binary_overlap = len(set(clustering_features) & set(supervised_binary_top))
    multi_overlap = len(set(clustering_features) & set(supervised_multi_top))
    
    print(f"Features in top 20 binary importance also used in clustering: {binary_overlap}/20")
    print(f"Features in top 20 multi-class importance also used in clustering: {multi_overlap}/20")
    
    # Create comparison visualization
    plt.figure(figsize=(20, 8))
    
    # Plot 1: Performance metrics
    plt.subplot(2, 2, 1)
    methods = ['Clustering ARI', 'Binary RF', 'Multi-class RF']
    scores = [clustering_results['ari'], supervised_results['binary_accuracy'], supervised_results['multi_accuracy']]
    bars = plt.bar(methods, scores, color=['lightblue', 'lightgreen', 'lightcoral'])
    plt.ylim(0, 1)
   
    plt.ylabel('Score')
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{score:.3f}', ha='center', va='bottom')
    
    # Plot 2: Feature overlap
    plt.subplot(2, 2, 2)
    overlap_types = ['Binary Overlap', 'Multi-class Overlap']
    overlap_counts = [binary_overlap, multi_overlap]
    bars = plt.bar(overlap_types, overlap_counts, color=['orange', 'purple'])
    plt.ylim(0, 20)
    plt.title('Feature Importance Overlap\n(Top 20 features)')
    plt.ylabel('Number of Overlapping Features')
    for bar, count in zip(bars, overlap_counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
                f'{count}/20', ha='center', va='bottom')

    # binary vs classification accuracy
    plt.subplot(2, 2, 4)
    comparison_data = [
        ('Binary Accuracy', supervised_results['binary_accuracy']),
        ('Multi-class Accuracy', supervised_results['multi_accuracy'])
    ]
    labels, values = zip(*comparison_data)
    bars = plt.bar(labels, values, color=['lightblue', 'lightgreen', 'lightcoral'])
    plt.ylim(0, 1)
    plt.title('Method Comparison')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{value:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'method_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nFindings:")
    print(f"- Clustering ARI: {clustering_results['ari']:.3f}")
    print(f"- Multi-class accuracy: {supervised_results['multi_accuracy']:.3f}")

In [ ]:
print("Creating balanced dataset across all ABF files")
all_features = []

for file_pair in tqdm(file_pairs, desc="Processing ABF files"):
    features = create_balanced_dataset_for_abf(
        file_pair['abf_file'],
        file_pair['classification_csv']
    )
    
    if features is not None:
        all_features.append(features)

combined_features = pd.concat(all_features, ignore_index=True)
print(f"\nCombined dataset: {len(combined_features)} total segments")
print(f"Class distribution:")
print(combined_features['analyte_label'].value_counts())

In [ ]:
from sklearn.feature_selection import VarianceThreshold, f_classif

'''Dropping features with greater than 30% NaN values'''
nan_threshold = 0.3 * len(combined_features)
combined_features.replace([np.inf, -np.inf], np.nan, inplace=True)

final_feature_df = combined_features.dropna(axis=1, thresh=nan_threshold)

# '''Dropping features with variance lower than 0,1'''
# numeric_features = final_feature_df.select_dtypes(include=['number'])
# selector = VarianceThreshold(threshold=0.001)
# reduced_features = selector.fit_transform(numeric_features)
# retained_columns = numeric_features.columns[selector.get_support()]
# final_feature_df = final_feature_df[retained_columns.tolist() + ['analyte_label']]  # Keep label

# '''Filling remaining NaNs with median of that feature'''
# final_feature_df.loc[:, final_feature_df.columns != 'analyte_label'] = final_feature_df.loc[:, final_feature_df.columns != 'analyte_label'].fillna(final_feature_df.median(numeric_only=True))
# final_feature_df = final_feature_df.dropna()

# '''For the purpose of clustering, I unite all Peptides Ladders to a single label'''
# final_feature_df["analyte_label"] = [col if 'ladder' not in col else 'ladder' for col in final_feature_df["analyte_label"]]


final_feature_df = final_feature_df.loc[:, ~final_feature_df.columns.str.contains('MFCC')]
final_feature_df = final_feature_df.loc[:, ~final_feature_df.columns.str.contains('signal_ECDF')]


final_feature_df = final_feature_df.drop(columns=[ 
    'signal_Interquartile range',
    'signal_Min',
    'signal_Max',
    'signal_Median frequency', 
    'signal_Centroid', 
    'signal_Spectral centroid',
    'signal_Standard deviation',
    'signal_Variance',
    'signal_Standard deviation',
    'signal_Median frequency', 
    "signal_Root mean square", 
    "signal_Average power", 
    "signal_Histogram mode", 
    "signal_Mean", 
    "signal_Median",
    "signal_Median frequency"])

In [ ]:
# final_feature_df.loc[final_feature_df['analyte_label'] != 'baseline', 'analyte_label'] = 'event'
# final_feature_df.loc[final_feature_df['true_label'] != 'baseline', 'true_label'] = 'event'
final_feature_df.loc[~final_feature_df['analyte_label'].str.startswith('b', na=False), 'analyte_label'] = 'event'
final_feature_df.loc[~final_feature_df['true_label'].str.startswith('b', na=False), 'true_label'] = 'event'


In [ ]:
print("\nstratified sampling for balance")
balanced_features = stratified_sample_by_analyte(final_feature_df, max_per_analyte=3000)
print(f"After sampling: {len(balanced_features)} segments")
print(f"Balanced distribution:")
print(balanced_features['analyte_label'].value_counts())

# Save the balanced dataset
balanced_features.to_csv(output_dir / 'balanced_analyte_features.csv', index=False)

# clustering analysis
clustering_results = perform_clustering_analysis(balanced_features)

# supervised analysis
supervised_results = perform_supervised_analysis(balanced_features)

# comparisons
compare_clustering_supervised_results(clustering_results, supervised_results)

supervised_results['binary_importance'].to_csv(output_dir / 'binary_feature_importance.csv', index=False)
supervised_results['multi_importance'].to_csv(output_dir / 'multiclass_feature_importance.csv', index=False)

# Save summary
summary = {
    'total_segments': len(balanced_features),
    'n_abf_files': len(file_pairs),
    'clustering_ari': float(clustering_results['ari']),
    'binary_accuracy': float(supervised_results['binary_accuracy']),
    'multiclass_accuracy': float(supervised_results['multi_accuracy']),
    'cv_mean_accuracy': float(supervised_results['cv_scores'].mean()),
    'cv_std_accuracy': float(supervised_results['cv_scores'].std()),
    'analytes_analyzed': balanced_features['analyte_label'].unique().tolist()
}

import json
with open(output_dir / 'analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"results saved to: {output_dir}")

In [ ]:
mask1 = final_feature_df['true_label'].str.startswith('b', na=False)
final_feature_df.loc[mask1, 'true_label'] = (
    final_feature_df.loc[mask1, 'true_label']
    .replace({
        r'.*Ladder.*': 'ladder',
        r'.*aplysia.*': 'mdr/aplysia',
        r'.*MDR.*': 'mdr/aplysia',
        r'.*AeL.*': 'h4ac'
    }, regex=True)
)
mask2 = final_feature_df['analyte_label'].str.startswith('b', na=False)
final_feature_df.loc[mask2, 'analyte_label'] = (
    final_feature_df.loc[mask2, 'analyte_label']
    .replace({
        r'.*Ladder.*': 'ladder',
        r'.*aplysia.*': 'mdr/aplysia',
        r'.*MDR.*': 'mdr/aplysia',
        r'.*AeL.*': 'h4ac'
    }, regex=True)
)

In [ ]:
final_feature_df

# Dropping all mean-related features

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Get only signal features
signal_cols = [col for col in balanced_features.columns if col.startswith('signal_')]
X = balanced_features[signal_cols].copy()
y = balanced_features['analyte_label']

# Ensure y is clean string labels
y = y.astype(str).str.strip()

# Remove any rows with weird labels
valid_labels = ['baseline', 'event']
mask = y.isin(valid_labels)
X = X[mask]
y = y[mask]

print(f"After cleaning - X shape: {X.shape}, y unique: {y.unique()}")

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

X = X.select_dtypes(include=[np.number])
X = X.fillna(0)

print(f"Final - X shape: {X.shape}")


model = RandomForestClassifier(n_estimators=100, random_state=42)
y_encoded = LabelEncoder().fit_transform(y)
model.fit(X, y_encoded)

print(f"Classes: {model.classes_}")

# SHAP WITH SAMPLING
# Use smaller sample to avoid memory issues
sample_size = min(500, len(X))
sample_idx = np.random.choice(len(X), sample_size, replace=False)
X_sample = X.iloc[sample_idx]

# SHAP with proper error handling
try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    print(f"SHAP type: {type(shap_values)}")
    if isinstance(shap_values, list):
        print(f"List length: {len(shap_values)}")
        for i, sv in enumerate(shap_values):
            print(f"  Class {i}: {sv.shape}")
    else:
        print(f"Array shape: {shap_values.shape}")
        
    # PLOT
    plt.figure(figsize=(12, 8))
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap.summary_plot(shap_values[1], X_sample, show=False, max_display=15)
    elif hasattr(shap_values, 'shape') and len(shap_values.shape) == 3:
        shap.summary_plot(shap_values[:, :, 1], X_sample, show=False, max_display=15)
    else:
        shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
    
    plt.title("SHAP Values - Events vs Baseline")
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"SHAP failed: {e}")
    print("Using feature importance instead...")
    
    importance = model.feature_importances_
    top_idx = np.argsort(importance)[-15:][::-1]
    
    plt.figure(figsize=(10, 8))
    plt.barh(range(15), importance[top_idx])
    plt.yticks(range(15), [X.columns[i] for i in top_idx])
    plt.gca().invert_yaxis()
    plt.xlabel('Feature Importance')
    plt.title('Top 15 Features')
    plt.tight_layout()
    plt.show()

In [ ]:
# Make the plot wider
plt.figure(figsize=(16, 10))  # Wider than before (was 12, 8)

if isinstance(shap_values, list) and len(shap_values) == 2:
    shap.summary_plot(shap_values[1], X_sample, show=False, max_display=20)
elif hasattr(shap_values, 'shape') and len(shap_values.shape) == 3:
    shap.summary_plot(shap_values[:, :, 1], X_sample, show=False, max_display=20)
else:
    shap.summary_plot(shap_values, X_sample, show=False, max_display=20)

plt.title("SHAP Values - Events vs Baseline", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(left=0.3)  # More space for feature names
plt.show()

In [ ]:
X = final_feature_df.select_dtypes(include=[np.number])
X = X.loc[:, X.notna().all(axis=0)]
X = X.loc[:, X.var(axis=0) > 1e-12]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

true_labels = final_feature_df['analyte_label']
le = LabelEncoder()
y_true = le.fit_transform(true_labels)

# --- KMeans clustering ---
kmeans = KMeans(n_clusters=len(np.unique(y_true)), random_state=42)
y_kmeans = kmeans.fit_predict(X_scaled)

# --- UMAP embedding ---
reducer = umap.UMAP(n_neighbors=150, min_dist=0.5, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# --- Plot UMAP colored by true labels ---
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=true_labels, palette="tab10", s=10)
plt.title("UMAP colored by True Labels")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# --- Plot UMAP colored by KMeans clusters ---
plt.subplot(1,2,2)
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=y_kmeans, palette="tab10", s=10)
plt.title("UMAP colored by KMeans Clusters")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
ari = adjusted_rand_score(y_true, y_kmeans)
nmi = normalized_mutual_info_score(y_true, y_kmeans)

print(f"Adjusted Rand Index (ARI): {ari:.3f}")
print(f"Normalized Mutual Information (NMI): {nmi:.3f}")